# Stacking y ensembles de modelos guardados

Este notebook reutiliza los modelos ya entrenados y guardados en `models/supervised/retrained_best_models/`. Así no hace falta repetir el Grid Search ni reentrenar los modelos base.

Se prueban dos enfoques:

- **Stacking preentrenado**: usa las probabilidades de los modelos guardados y entrena una regresión logística como metamodelo.
- **Ensemble ponderado**: combina probabilidades con distintos pesos para ver si una media ponderada mejora el ranking.

Nota metodológica: al usar modelos base ya entrenados, este stacking puede ser más optimista que un stacking con predicciones out-of-fold. Por eso se interpreta como experimento adicional, no como sustituto automático del mejor modelo individual.

## Librerías y configuración

In [1]:
from itertools import product
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_JOBS = -1

## Carga de datos

Se reconstruye el mismo split estratificado usado en el notebook de modelos para poder evaluar los modelos guardados en el mismo conjunto de test.

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_root / "data" / "clean"
models_path = project_root / "models"
retrained_models_path = models_path / "supervised" / "retrained_best_models"

X = pd.read_csv(data_path / "X_clean.csv")
y = pd.read_csv(data_path / "y.csv").squeeze("columns")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Fraude en test:", round(y_test.mean() * 100, 3), "%")

X_train: (475714, 10)
X_test: (118929, 10)
Fraude en test: 1.211 %


## Funciones de evaluación

In [3]:
def format_seconds(seconds):
    minutes, sec = divmod(float(seconds), 60)
    hours, minutes = divmod(int(minutes), 60)
    if hours:
        return f"{hours}h {minutes:02d}m {sec:04.1f}s"
    if minutes:
        return f"{minutes}m {sec:04.1f}s"
    return f"{sec:.1f}s"


def threshold_metrics(y_true, y_proba, beta=2):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    beta2 = beta ** 2
    scores = (1 + beta2) * precision * recall / ((beta2 * precision) + recall + 1e-12)
    best_idx = int(np.nanargmax(scores))
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 1.0
    y_pred = (y_proba >= best_threshold).astype(int)

    return {
        "best_threshold": best_threshold,
        "f2_tuned": fbeta_score(y_true, y_pred, beta=2),
        "classification_report_tuned": classification_report(
            y_true,
            y_pred,
            target_names=["legitima", "fraude"],
        ),
        "confusion_matrix_tuned": confusion_matrix(y_true, y_pred),
    }


def evaluate_proba(name, y_true, y_proba, y_pred=None, fit_time_sec=0.0, extra=None):
    extra = extra or {}
    if y_pred is None:
        y_pred = (y_proba >= 0.5).astype(int)

    metrics = {
        "model": name,
        "algorithm": extra.get("algorithm", "Ensemble"),
        "smote": extra.get("smote", "mixto"),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "f1": fbeta_score(y_true, y_pred, beta=1),
        "f2": fbeta_score(y_true, y_pred, beta=2),
        "fit_time_sec": fit_time_sec,
        "best_params": extra,
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "classification_report": classification_report(
            y_true,
            y_pred,
            target_names=["legitima", "fraude"],
        ),
    }
    metrics.update(threshold_metrics(y_true, y_proba, beta=2))
    return metrics


def summarize_results(results):
    table = pd.DataFrame(results).T
    if "fit_time_sec" in table.columns:
        table["fit_time_sec"] = table["fit_time_sec"].astype(float)
        table["fit_time_min"] = table["fit_time_sec"] / 60
        table["fit_time"] = table["fit_time_sec"].map(format_seconds)

    metric_cols = [
        "roc_auc", "pr_auc", "f1", "f2", "f2_tuned",
        "best_threshold", "fit_time_sec", "fit_time_min",
    ]
    existing = [col for col in metric_cols if col in table.columns]
    table[existing] = table[existing].astype(float).round(4)
    return table.sort_values("pr_auc", ascending=False)

## Carga de modelos guardados

In [4]:
model_files = {
    "LightGBM_con_smote": retrained_models_path / "LightGBM_con_smote.joblib",
    "LightGBM_sin_smote": retrained_models_path / "LightGBM_sin_smote.joblib",
    "XGBoost_sin_smote": retrained_models_path / "XGBoost_sin_smote.joblib",
    "RandomForest_sin_smote": retrained_models_path / "RandomForest_sin_smote.joblib",
}

missing = [str(path) for path in model_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Faltan modelos guardados. Ejecuta primero el reentrenamiento en 04_Modelos_final. "
        f"No encontrados: {missing}"
    )

loaded_models = {name: joblib.load(path) for name, path in model_files.items()}

print("Modelos cargados:")
for name, path in model_files.items():
    print(f"- {name}: {path}")

Modelos cargados:
- LightGBM_con_smote: /Users/aldan/Desktop/tfg_code/models/retrained_best_models/LightGBM_con_smote.joblib
- LightGBM_sin_smote: /Users/aldan/Desktop/tfg_code/models/retrained_best_models/LightGBM_sin_smote.joblib
- XGBoost_sin_smote: /Users/aldan/Desktop/tfg_code/models/retrained_best_models/XGBoost_sin_smote.joblib
- RandomForest_sin_smote: /Users/aldan/Desktop/tfg_code/models/retrained_best_models/RandomForest_sin_smote.joblib


## Evaluación de modelos base cargados

In [5]:
base_results = {}
base_probas_test = {}
base_probas_train = {}

metadata = {
    "LightGBM_con_smote": {"algorithm": "LightGBM", "smote": True},
    "LightGBM_sin_smote": {"algorithm": "LightGBM", "smote": False},
    "XGBoost_sin_smote": {"algorithm": "XGBoost", "smote": False},
    "RandomForest_sin_smote": {"algorithm": "RandomForest", "smote": False},
}

for name, model in loaded_models.items():
    y_proba_test = model.predict_proba(X_test)[:, 1]
    y_pred_test = model.predict(X_test)
    y_proba_train = model.predict_proba(X_train)[:, 1]

    base_probas_test[name] = y_proba_test
    base_probas_train[name] = y_proba_train
    base_results[name] = evaluate_proba(
        name,
        y_test,
        y_proba_test,
        y_pred=y_pred_test,
        extra=metadata.get(name, {}),
    )

base_results_df = summarize_results(base_results)
display(
    base_results_df[
        ["model", "algorithm", "smote", "roc_auc", "pr_auc", "f1", "f2", "f2_tuned", "best_threshold"]
    ]
)

,model,algorithm,smote,roc_auc,pr_auc,f1,f2,f2_tuned,best_threshold
LightGBM_con_smote,LightGBM_con_smote,LightGBM,True,0.9984,0.9269,0.8542,0.8433,0.8659,0.2646
LightGBM_sin_smote,LightGBM_sin_smote,LightGBM,False,0.9985,0.9266,0.8510,0.8240,0.8692,0.1304
XGBoost_sin_smote,XGBoost_sin_smote,XGBoost,False,0.9981,0.9121,0.5719,0.7601,0.8548,0.9085
RandomForest_sin_smote,RandomForest_sin_smote,RandomForest,False,0.9972,0.8966,0.6788,0.8100,0.8348,0.7493


## Stacking preentrenado

`StackingClassifier(cv="prefit")` usa los modelos ya ajustados y entrena solo el metamodelo. Es rápido, pero puede ser optimista porque los modelos base ya vieron `X_train`. Se incluye como experimento comparativo.

In [6]:
stacking_base_models = [
    "LightGBM_con_smote",
    "XGBoost_sin_smote",
    "RandomForest_sin_smote",
]

stacking_estimators = [(name, loaded_models[name]) for name in stacking_base_models]

stacking_prefit = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=LogisticRegression(
        C=0.5,
        penalty="l2",
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    stack_method="predict_proba",
    cv="prefit",
    n_jobs=N_JOBS,
    passthrough=False,
)

stacking_prefit.fit(X_train, y_train)
y_proba_stacking = stacking_prefit.predict_proba(X_test)[:, 1]

stacking_results = {
    "Stacking_prefit_diverse": evaluate_proba(
        "Stacking_prefit_diverse",
        y_test,
        y_proba_stacking,
        extra={
            "algorithm": "Stacking",
            "smote": "mixto",
            "base_models": stacking_base_models,
            "note": "cv='prefit'; interpretación exploratoria por posible optimismo.",
        },
    )
}

display(
    summarize_results(stacking_results)[
        ["model", "algorithm", "smote", "roc_auc", "pr_auc", "f1", "f2", "f2_tuned", "best_threshold"]
    ]
)

,model,algorithm,smote,roc_auc,pr_auc,f1,f2,f2_tuned,best_threshold
Stacking_prefit_diverse,Stacking_prefit_diverse,Stacking,mixto,0.9982,0.9213,0.8502,0.8413,0.8577,0.2374


## Ensemble ponderado de probabilidades

Este enfoque no entrena modelos nuevos: solo combina probabilidades. Es útil para comprobar si los modelos aportan señales complementarias. Los pesos se seleccionan sobre test, por lo que debe interpretarse como diagnóstico, no como estimación final insesgada.

In [7]:
weight_grid = [0.0, 0.25, 0.5, 0.75, 1.0]
weighted_candidates = stacking_base_models
weighted_results_search = []

for weights in product(weight_grid, repeat=len(weighted_candidates)):
    if sum(weights) == 0:
        continue
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()

    y_proba_weighted = sum(
        weight * base_probas_test[name]
        for weight, name in zip(weights, weighted_candidates)
    )

    weighted_results_search.append(
        {
            "weights": dict(zip(weighted_candidates, weights.round(3))),
            "pr_auc": average_precision_score(y_test, y_proba_weighted),
            "roc_auc": roc_auc_score(y_test, y_proba_weighted),
            "y_proba": y_proba_weighted,
        }
    )

best_weighted = max(weighted_results_search, key=lambda row: row["pr_auc"])
weighted_results = {
    "Weighted_probability_ensemble": evaluate_proba(
        "Weighted_probability_ensemble",
        y_test,
        best_weighted["y_proba"],
        extra={
            "algorithm": "WeightedEnsemble",
            "smote": "mixto",
            "weights": best_weighted["weights"],
            "note": "Pesos seleccionados sobre test; diagnóstico, no estimación final insesgada.",
        },
    )
}

print("Mejores pesos:", best_weighted["weights"])
display(
    summarize_results(weighted_results)[
        ["model", "algorithm", "smote", "roc_auc", "pr_auc", "f1", "f2", "f2_tuned", "best_threshold"]
    ]
)

Mejores pesos: {'LightGBM_con_smote': np.float64(1.0), 'XGBoost_sin_smote': np.float64(0.0), 'RandomForest_sin_smote': np.float64(0.0)}


,model,algorithm,smote,roc_auc,pr_auc,f1,f2,f2_tuned,best_threshold
Weighted_probability_ensemble,Weighted_probability_ensemble,WeightedEnsemble,mixto,0.9984,0.9269,0.8542,0.8433,0.8659,0.2646


## Comparativa final y guardado

In [8]:
all_results = {**base_results, **stacking_results, **weighted_results}
comparison_df = summarize_results(all_results)

comparison_cols = [
    "model", "algorithm", "smote", "roc_auc", "pr_auc",
    "f1", "f2", "f2_tuned", "best_threshold", "fit_time",
]
comparison_cols = [col for col in comparison_cols if col in comparison_df.columns]
display(comparison_df[comparison_cols])

best_name = comparison_df.index[0]
print("Mejor modelo/ensemble por PR-AUC:", best_name)
print("PR-AUC:", comparison_df.loc[best_name, "pr_auc"])
print("F2 ajustado:", comparison_df.loc[best_name, "f2_tuned"])

print("\nClassification report con umbral ajustado del mejor:")
print(all_results[best_name]["classification_report_tuned"])
print("Matriz de confusión con umbral ajustado:")
print(all_results[best_name]["confusion_matrix_tuned"])

output_df = comparison_df.drop(
    columns=[
        "confusion_matrix", "classification_report",
        "confusion_matrix_tuned", "classification_report_tuned",
    ],
    errors="ignore",
)
output_path = models_path / "reports" / "model_comparison_06_stacking.csv"
output_df.to_csv(output_path, index=True)

stacking_path = models_path / "stacking" / "stacking_prefit_fraud_pipeline.joblib"
joblib.dump(stacking_prefit, stacking_path)

print("Comparativa guardada en:", output_path)
print("Stacking guardado en:", stacking_path)

,model,algorithm,smote,roc_auc,pr_auc,f1,f2,f2_tuned,best_threshold,fit_time
LightGBM_con_smote,LightGBM_con_smote,LightGBM,True,0.9984,0.9269,0.8542,0.8433,0.8659,0.2646,0.0s
Weighted_probability_ensemble,Weighted_probability_ensemble,WeightedEnsemble,mixto,0.9984,0.9269,0.8542,0.8433,0.8659,0.2646,0.0s
LightGBM_sin_smote,LightGBM_sin_smote,LightGBM,False,0.9985,0.9266,0.8510,0.8240,0.8692,0.1304,0.0s
Stacking_prefit_diverse,Stacking_prefit_diverse,Stacking,mixto,0.9982,0.9213,0.8502,0.8413,0.8577,0.2374,0.0s
XGBoost_sin_smote,XGBoost_sin_smote,XGBoost,False,0.9981,0.9121,0.5719,0.7601,0.8548,0.9085,0.0s
RandomForest_sin_smote,RandomForest_sin_smote,RandomForest,False,0.9972,0.8966,0.6788,0.8100,0.8348,0.7493,0.0s


Mejor modelo/ensemble por PR-AUC: LightGBM_con_smote
PR-AUC: 0.9269
F2 ajustado: 0.8659

Classification report con umbral ajustado del mejor:
              precision    recall  f1-score   support

    legitima       1.00      1.00      1.00    117489
      fraude       0.78      0.89      0.83      1440

    accuracy                           1.00    118929
   macro avg       0.89      0.94      0.91    118929
weighted avg       1.00      1.00      1.00    118929

Matriz de confusión con umbral ajustado:
[[117128    361]
 [   158   1282]]
Comparativa guardada en: /Users/aldan/Desktop/tfg_code/models/model_comparison_06_stacking.csv
Stacking guardado en: /Users/aldan/Desktop/tfg_code/models/stacking_prefit_fraud_pipeline.joblib


## Interpretación

Si `Stacking_prefit_diverse` o `Weighted_probability_ensemble` no superan a `LightGBM_con_smote` en PR-AUC, la conclusión es que el mejor modelo individual sigue siendo preferible. Si solo mejora `f2_tuned`, la mejora viene principalmente del ajuste de umbral, no de un ranking más potente.

Para el TFG, lo más defendible es presentar este notebook como experimento de ensemble posterior al entrenamiento principal.